In [ ]:
import sympy as sp
import numpy as np
from IPython.display import Math

import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
import Plot_setting
from Plot_setting import *
from Utils import * 

import matplotlib.pyplot as plt


In [ ]:

# Define your symbol
a = sp.symbols('a', complex=True)
a_st = sp.symbols('a^*', complex=True)
b = sp.symbols('b', complex = True)
b_st = sp.symbols('b^*', complex=True)

w_p = sp.symbols('\\omega_p', real =True)
w_d = sp.symbols('\\omega_d', real =True)
da =  sp.symbols('\\delta_a', real =True)
db =  sp.symbols('\\delta_b', real =True)
xi = sp.symbols('\\xi', complex= True)
r = sp.symbols('r', real = True)
t = sp.symbols('t', real =True)

eps = sp.symbols('\\varepsilon', real = True)

# Rotating frame expressions
rot_a = sp.exp(-sp.I * (w_p + w_d) * t / 2)
rot_b = sp.exp(-sp.I * w_d * t)

A = sp.symbols('A', complex = True)
A_st = sp.symbols('A^*', complex = True)
B = sp.symbols('B', complex = True)
B_st = sp.symbols('B^*', complex = True)

In [ ]:
def degree(expr:sp.Expr, x:sp.Expr):
    """
    Returns the degree of expr wrt to x
    :param expr: expression
    :param x: variable
    :return: degree
    """
    if type(expr) == int : return 0
    n=1
    while expr.has(x**n):
        n+=1
    return n-1

In [ ]:
def x_indep_terms(expr: sp.Expr, x: sp.Expr):
    """
    Returns the terms that do not depend on x in the expr that should be in an expanded form.
    :param expr: expression that is expanded already
    :param x: variable
    """
    
    exp2 = 0
    exps = expr + 1  # ensures exps is of type Add even if expr is a monomial
    for e in exps.args:
        if e.has(x):
            e = 0
        exp2 += e
    return exp2-1

def poisson_bracket(f,g,variables):
    return(sp.diff(f,variables[0])*sp.diff(g,variables[1])-sp.diff(g,variables[0])*sp.diff(f,variables[1]))

def Hamiltonian(n,k_max):
    # Coupling coefficients: g[i, j]
    g = sp.Matrix((n+1)//2, (k_max+1)//2, lambda i, j: sp.symbols(f"g_{2*i+1}_{2*j+1}", real=True))



    # Define the summand integrand
    summand_int = a * rot_a + a_st * sp.conjugate(rot_a) + r * ((b + xi) * rot_b + (b_st + xi) * sp.conjugate(rot_b))

    # Build the matrix of summands
    summand = [
        [summand_int**n_i * (sp.exp(sp.I*k * w_p * t)+sp.exp(-sp.I*k * w_p * t)) for k in range(1, k_max + 1, 2)]
        for n_i in range(1, n + 1, 2)
    ]
    H = eps*da*a_st*a+eps*db*b_st*b+sum([eps**(2*i+1) * g[i, j] * summand[i][j] for i in range((n+1)//2) for j in range((k_max+1)//2)])
    return(H)

def transformation(eta, mu, K):
    n = degree(eta, eps)
    eta += eps**(n+1) * sp.Function(f'eta^({n+1})', complex=True)(A,A_st,B,B_st,t)
    eta_st = sp.conjugate(eta)

    a_new = A+eta
    a_st_new = A_st+eta_st


    mu += eps**(n+1) * sp.Function(f'mu^({n+1})', complex=True)(A,A_st,B,B_st,t)
    mu_st = sp.conjugate(mu)

    b_new = B+mu
    b_st_new = B_st+mu_st

    K += eps**(n+1) * sp.Function(f'K^({n+1})', complex=True)(A,A_st,B,B_st)
        
    return(a_new,a_st_new,b_new,b_st_new,eta,eta_st,mu,mu_st,K)

def formating_rotating(O: sp.Expr):
    """
    This function takes an operator O and returns the same operator formated such that 
    it takes the form: O = sum_{i} O_i e^{sp.I (F(omega)) st}
    In particular it ensures that the exponentials are grouped together at the numerator and simplified.
    An optimal use of this method would be to perform the integration along with the formating.
    :param O: operator
    :return: formated operator
    """
    O_copy = 1 * O +1
    O_copy = sp.expand(O_copy, power_exp=False)
    Expr2 = 0*sp.I
    if type(O_copy) == sp.core.add.Add:

        for a in O_copy.args:
            Expr2 += sp.powsimp(sp.factor_terms(a), force=True)
        O_copy = Expr2
    return O_copy-1

def grouping_exponential(O: sp.Expr):
    """
    Group terms in O by their exponential frequency components e^{i * f(t)}.
    """
    frequencies_exp = list(O.atoms(sp.exp))
    copy = 0
    for f in frequencies_exp:
        copy += sp.simplify(x_indep_terms(sp.expand(O*sp.conjugate(f)), t))*f
    return copy

# First approach

In [ ]:
test  = B*rot_b + xi*rot_b
formating_rotating(test, t)

In [ ]:
list(test.atoms(sp.exp))

In [ ]:
test_args_norm = [arg/sp.Abs(arg) for arg in test.args]
test_args_norm

In [ ]:
sp.arg(rot_a)

In [ ]:
n_max = 3
k_max = 9

H_1 = Hamiltonian(1,k_max)
eta = 0
mu = 0
K = 0

a_new,a_st_new,b_new,b_st_new,eta,eta_st,mu,mu_st,K = transformation(eta, mu, K)

In [ ]:
K

In [ ]:
eq_expr_a = (sp.diff(H_1,a_st).replace(a, a_new)).replace(b,b_new).replace(a_st, a_st_new).replace(b_st,b_st_new) +sp.I*poisson_bracket(K,eta,(A,A_st))
eq_expr_a_st = (sp.diff(H_1,a).replace(a, a_new)).replace(b,b_new).replace(a_st, a_st_new).replace(b_st,b_st_new) +sp.I*poisson_bracket(K,eta_st,(A,A_st))

eq_expr_b = (sp.diff(H_1,b_st).replace(a, a_new)).replace(b,b_new).replace(a_st, a_st_new).replace(b_st,b_st_new) +sp.I*poisson_bracket(K,mu,(B,B_st))
eq_expr_b_st = (sp.diff(H_1,b).replace(a, a_new)).replace(b,b_new).replace(a_st, a_st_new).replace(b_st,b_st_new) +sp.I*poisson_bracket(K,mu_st,(B,B_st))

In [ ]:
first_order_a = x_indep_terms(sp.expand(eq_expr_a/eps), eps)*eps
first_order_a_st = x_indep_terms(sp.expand(eq_expr_a_st/eps), eps)*eps

first_order_b = x_indep_terms(sp.expand(eq_expr_b/eps), eps)*eps
first_order_b_st = x_indep_terms(sp.expand(eq_expr_b_st/eps), eps)*eps

In [ ]:
first_order_a

In [ ]:
d_A_st_K = x_indep_terms(sp.expand(first_order_a), t)
d_A_K = x_indep_terms(sp.expand(first_order_a_st), t)

d_B_st_K = x_indep_terms(sp.expand(first_order_b), t)
d_B_K = x_indep_terms(sp.expand(first_order_b_st), t)

In [ ]:
print(d_A_st_K)
print(d_A_K)
print(d_B_st_K)
print(d_B_K)

In [ ]:
K = eps * (da*A_st*A + da*B_st*B)

In [ ]:
d_t_eta = formating_rotating(sp.expand(first_order_a-d_A_st_K)*(-sp.I))
d_t_mu = formating_rotating(sp.expand(first_order_b-d_B_st_K)*(-sp.I))

In [ ]:
d_t_eta

In [ ]:
eta = d_t_eta.replace(sp.exp, lambda x: sp.exp(x) / x.subs(t, 1))  # Integration
mu = d_t_mu.replace(sp.exp, lambda x: sp.exp(x) / x.subs(t, 1))  # Integration

In [ ]:
a_new,a_st_new,b_new,b_st_new,eta,eta_st,mu,mu_st,K = transformation(eta, mu, K)
H_2 = Hamiltonian(2,k_max)

In [ ]:
H_2

In [ ]:
eta

In [ ]:
eq_expr_a = (sp.diff(H_2,a_st).replace(a, a_new)).replace(b,b_new).replace(a_st, a_st_new).replace(b_st,b_st_new) +sp.I*poisson_bracket(K,eta,(A,A_st))
eq_expr_a_st = (sp.diff(H_2,a).replace(a, a_new)).replace(b,b_new).replace(a_st, a_st_new).replace(b_st,b_st_new) +sp.I*poisson_bracket(K,eta_st,(A,A_st))

eq_expr_b = (sp.diff(H_2,b_st).replace(a, a_new)).replace(b,b_new).replace(a_st, a_st_new).replace(b_st,b_st_new) +sp.I*poisson_bracket(K,mu,(B,B_st))
eq_expr_b_st = (sp.diff(H_2,b).replace(a, a_new)).replace(b,b_new).replace(a_st, a_st_new).replace(b_st,b_st_new) +sp.I*poisson_bracket(K,mu_st,(B,B_st))

In [ ]:
second_order_a = x_indep_terms(sp.expand(eq_expr_a/(eps**2)), eps)*eps**2
second_order_a_st = x_indep_terms(sp.expand(eq_expr_a_st/(eps**2)), eps)*eps**2

second_order_b = x_indep_terms(sp.expand(eq_expr_b/(eps**2)), eps)*(eps**2)
second_order_b_st = x_indep_terms(sp.expand(eq_expr_b_st/(eps**2)), eps)*(eps**2)

In [ ]:
second_order_a

In [ ]:
d_A_st_K = x_indep_terms(sp.expand(second_order_a), t)
d_A_K = x_indep_terms(sp.expand(second_order_a_st), t)

d_B_st_K = x_indep_terms(sp.expand(second_order_b), t)
d_B_K = x_indep_terms(sp.expand(second_order_b_st), t)

In [ ]:
print(d_A_st_K)
print(d_A_K)
print(d_B_st_K)
print(d_B_K)

In [ ]:
K = eps * (da*A_st*A + da*B_st*B)

In [ ]:
d_t_eta = formating_rotating(sp.expand(second_order_a-d_A_st_K)*(-sp.I))
d_t_mu = formating_rotating(sp.expand(second_order_b-d_B_st_K)*(-sp.I))

In [ ]:
d_t_eta

In [ ]:
eta = eta-x_indep_terms(sp.expand(eta/(eps**2)), eps)*(eps**2)+d_t_eta.replace(sp.exp, lambda x: sp.exp(x) / x.subs(t, 1))  # Integration
mu = mu-x_indep_terms(sp.expand(mu/(eps**2)), eps)*(eps**2)+d_t_mu.replace(sp.exp, lambda x: sp.exp(x) / x.subs(t, 1))  # Integration

In [ ]:
K

In [ ]:
a_new,a_st_new,b_new,b_st_new,eta,eta_st,mu,mu_st,K = transformation(eta, mu, K)
H_3 = Hamiltonian(3,k_max)

In [ ]:
K

In [ ]:
eq_expr_a = (sp.diff(H_3,a_st).replace(a, a_new)).replace(b,b_new).replace(a_st, a_st_new).replace(b_st,b_st_new) +sp.I*poisson_bracket(K,eta,(A,A_st))
eq_expr_a_st = (sp.diff(H_3,a).replace(a, a_new)).replace(b,b_new).replace(a_st, a_st_new).replace(b_st,b_st_new) +sp.I*poisson_bracket(K,eta_st,(A,A_st))

eq_expr_b = (sp.diff(H_3,b_st).replace(a, a_new)).replace(b,b_new).replace(a_st, a_st_new).replace(b_st,b_st_new) +sp.I*poisson_bracket(K,mu,(B,B_st))
eq_expr_b_st = (sp.diff(H_3,b).replace(a, a_new)).replace(b,b_new).replace(a_st, a_st_new).replace(b_st,b_st_new) +sp.I*poisson_bracket(K,mu_st,(B,B_st))

In [ ]:
third_order_a = x_indep_terms(sp.expand(eq_expr_a/(eps**3)), eps)*eps**3
third_order_a_st = x_indep_terms(sp.expand(eq_expr_a_st/(eps**3)), eps)*eps**3

third_order_b = x_indep_terms(sp.expand(eq_expr_b/(eps**3)), eps)*(eps**3)
third_order_b_st = x_indep_terms(sp.expand(eq_expr_b_st/(eps**3)), eps)*(eps**3)

In [ ]:
eq_expr_a

In [ ]:
third_order_a

In [ ]:
d_A_st_K = x_indep_terms(sp.expand(third_order_a), t)
d_A_K = x_indep_terms(sp.expand(third_order_a_st), t)

d_B_st_K = x_indep_terms(sp.expand(third_order_b), t)
d_B_K = x_indep_terms(sp.expand(third_order_b_st), t)

In [ ]:
print(d_A_st_K)
print(d_A_K)
print(d_B_st_K)
print(d_B_K)

In [ ]:
sp.simplify(d_A_st_K)

In [ ]:
sp.simplify(d_A_K)

In [ ]:
sp.simplify(d_B_st_K)

In [ ]:
sp.simplify(d_B_K)

# New approach

In [ ]:
n_max=3
k_max=9

eta_funcs = [0]
eta_exprs = {0:0}
eta_simplified = {0:0}

mu_funcs = [0]
mu_exprs = {0:0}
mu_simplified = {0:0}

K_funcs = [0]
K_exprs = {0:0}
K_simplified = {0:0}

for i in range(1, n_max+1):
    eta = sp.Function(fr'eta^({i})', complex=True)(A, A_st, B, B_st, t)
    eta_simplified[sp.symbols(fr'eta^({i})', complex=True)] = eta
    eta_funcs.append(eta)

    mu = sp.Function(fr'mu^({i})', complex=True)(A, A_st, B, B_st, t)
    mu_simplified[sp.symbols(fr'mu^({i})', complex=True)] = mu
    mu_funcs.append(mu)

    K = sp.Function(fr'K^({i})', complex=True)(A, A_st, B, B_st)
    K_simplified[sp.symbols(fr'K^({i})', complex=True)] = K
    K_funcs.append(K)

g = sp.Matrix((n_max+1)//2, (k_max+1)//2, lambda i, j: sp.symbols(f"g_{2*i+1}_{2*j+1}", real=True))


In [ ]:
list(sp.symbols(f'Y0:{4}'))

In [ ]:
from scipy.special import binom
def get_ith_order_term(poly, var, power, i):
    """
    Returns the coefficient of x^i in (poly)^power.

    Parameters:
        poly : sympy expression (the base polynomial)
        var : sympy.Symbol (the variable)
        power : int (exponent)
        i : int (order of the term to extract)

    Returns:
        sympy expression: the coefficient of x^i
    """
    expanded = sp.expand(poly**power)
    return expanded.coeff(var, i)

def eq_a_n(n):
    term_1 = da*eta_funcs[n-1]
    if n == 1:
        term_1 += da*A
    term_2 = 0
    for i in range(1,n):
        term_2 += poisson_bracket(K_funcs[i], eta_funcs[n-i], (A,A_st))
    
    term_3 = 0
    X = sp.symbols('X', complex=True)
    # Create symbols Y_0, ..., Y_n
    Y = list(sp.symbols(f'Y0:{n}'))
    Y[0]=0
    Polynom = sum(Y[i] * eps**(i) for i in range(n))
    for i in range(1, n+1, 2):
        for l in range(0, i):
            #print(i,l, Polynom, get_ith_order_term(Polynom, eps, l, n-i))
            term_3 += binom(i-1,l)*(X**(i-1-l))*get_ith_order_term(Polynom, eps, l, n-i)
            #print(term_3)
        term_3 *=sum((sp.exp(sp.I*k * w_p * t)+sp.exp(-sp.I*k * w_p * t))*g[(i-1)//2,(k-1)//2] for k in range(1, k_max + 1, 2))
        term_3 *=i
    
    term_3 *= rot_a
    X_expr = A * rot_a + A_st * sp.conjugate(rot_a) + r * ((B + xi) * rot_b + (B_st + xi) * sp.conjugate(rot_b))
    term_3 = term_3.replace(X, X_expr)

    for i in range(0, n):
        Y_expr = eta_funcs[i]*rot_a + sp.conjugate(eta_funcs[i]) * sp.conjugate(rot_a) + r * (mu_funcs[i] * rot_b + sp.conjugate(mu_funcs[i]) * sp.conjugate(rot_b))
        term_3 = term_3.replace(Y[i], Y_expr)

    result = term_1 + term_2 + term_3
    return result

def eq_b_n(n):
    term_1 = db*mu_funcs[n-1]
    if n == 1:
        term_1 += db*B
    term_2 = 0
    for i in range(1,n):
        term_2 += poisson_bracket(K_funcs[i], mu_funcs[n-i], (B,B_st))
    
    term_3 = 0
    X = sp.symbols('X', complex=True)
    # Create symbols Y_0, ..., Y_n
    Y = list(sp.symbols(f'Y0:{n}'))
    Y[0]=0
    Polynom = sum(Y[i] * eps**(i) for i in range(n))
    for i in range(1, n+1, 2):
        for l in range(0, i):
            #print(i,l, Polynom, get_ith_order_term(Polynom, eps, l, n-i))
            term_3 += binom(i-1,l)*(X**(i-1-l))*get_ith_order_term(Polynom, eps, l, n-i)
            #print(term_3)
        term_3 *=sum((sp.exp(sp.I*k * w_p * t)+sp.exp(-sp.I*k * w_p * t))*g[(i-1)//2,(k-1)//2] for k in range(1, k_max + 1, 2))
        term_3 *=i
    
    term_3 *= rot_b
    X_expr = A * rot_a + A_st * sp.conjugate(rot_a) + r * ((B + xi) * rot_b + (B_st + xi) * sp.conjugate(rot_b))
    term_3 = term_3.replace(X, X_expr)

    for i in range(0, n):
        Y_expr = eta_funcs[i]*rot_a + sp.conjugate(eta_funcs[i]) * sp.conjugate(rot_a) + r * (mu_funcs[i] * rot_b + sp.conjugate(mu_funcs[i]) * sp.conjugate(rot_b))
        term_3 = term_3.replace(Y[i], Y_expr)

    result = term_1 + term_2 + term_3
    return result

def eq_n(n, mode):
    if mode == 'a':
        greek_funcs = eta_funcs
        greek_exprs = eta_exprs
        delta = da
        capital = A
        capital_st = A_st
        rotation = sp.conjugate(rot_a)
    elif mode == 'b':
        greek_funcs = mu_funcs
        greek_exprs = mu_exprs
        delta = db
        capital = B
        capital_st = B_st
        rotation = sp.conjugate(rot_b)*r
    term_1 = delta*greek_exprs[greek_funcs[n-1]]
    if n == 1:
        term_1 += delta*capital
    term_2 = 0
    for i in range(1,n):
        term_2 += poisson_bracket(K_exprs[K_funcs[i]], greek_exprs[greek_funcs[n-i]], (capital,capital_st))

    term_3 = 0
    X = sp.symbols('X', complex=True)
    # Create symbols Y_0, ..., Y_n
    Y = list(sp.symbols(f'Y0:{n}'))
    Y[0]=0
    Polynom = sum(Y[i] * eps**(i) for i in range(n))
    for i in range(1, n+1, 2):
        for l in range(0, i):
            #print(i,l, Polynom, get_ith_order_term(Polynom, eps, l, n-i))
            term_3 += binom(i-1,l)*(X**(i-1-l))*get_ith_order_term(Polynom, eps, l, n-i)
            #print(term_3)
        term_3 *=sum((sp.exp(sp.I*k * w_p * t)+sp.exp(-sp.I*k * w_p * t))*g[(i-1)//2,(k-1)//2] for k in range(1, k_max + 1, 2))
        term_3 *=i
    
    term_3 *= rotation
    X_expr = A * rot_a + A_st * sp.conjugate(rot_a) + r * ((B + xi) * rot_b + (B_st + xi) * sp.conjugate(rot_b))
    term_3 = term_3.replace(X, X_expr)

    for i in range(0, n):
        Y_expr = eta_exprs[eta_funcs[i]]*rot_a + sp.conjugate(eta_exprs[eta_funcs[i]]) * sp.conjugate(rot_a) + r * (mu_exprs[mu_funcs[i]] * rot_b + sp.conjugate(mu_exprs[mu_funcs[i]]) * sp.conjugate(rot_b))
        term_3 = term_3.replace(Y[i], Y_expr)

    result = term_1 + term_2 + term_3
    return result

In [ ]:
for n in range(1, n_max+1):
    eq_a = eq_n(n, 'a')
    eq_b = eq_n(n, 'b')
    
    d_A_st_K = x_indep_terms(sp.expand(eq_a), t)
    d_A_K = sp.conjugate(d_A_st_K).replace(sp.conjugate(A),A_st).replace(sp.conjugate(B),B_st).replace(sp.conjugate(A_st),A).replace(sp.conjugate(B_st),B)

    d_B_st_K = x_indep_terms(sp.expand(eq_b), t)
    d_B_K = sp.conjugate(d_B_st_K).replace(sp.conjugate(A),A_st).replace(sp.conjugate(B),B_st).replace(sp.conjugate(A_st),A).replace(sp.conjugate(B_st),B)
    
    K_0 = sp.integrate(d_A_st_K, A_st) 
    d_A_K_0 = sp.diff(K_0, A)
    res1 = sp.expand(d_A_K - d_A_K_0)
    if res1.has(A_st):
        raise ValueError("Not integrable: compatibility condition failed (res1 depends on A_st).")
    K_0 += sp.integrate(res1, A)

    d_B_st_K_0 = sp.diff(K_0, B_st)
    res2 = sp.expand(d_B_st_K - d_B_st_K_0)
    if res2.has(A) or res2.has(A_st):
        raise ValueError("Not integrable: compatibility condition failed (res2 depends on A_st or A).")
    K_0 += sp.integrate(res2, B_st)

    d_B_K_0 = sp.diff(K_0, B)
    res3 = sp.expand(d_B_K - d_B_K_0)
    if res3.has(A) or res3.has(A_st) or res3.has(B_st):
        raise ValueError("Not integrable: compatibility condition failed (res3 depends on A_st or A or B_st).")
    K_0 += sp.integrate(res3, B)

    K_exprs[K_funcs[n]]=sp.simplify(K_0)


    d_t_eta = formating_rotating(sp.expand(eq_a-d_A_st_K)*(-sp.I))
    d_t_mu = formating_rotating(sp.expand(eq_b-d_B_st_K)*(-sp.I))

    eta_exprs[eta_funcs[n]] = (d_t_eta.replace(sp.exp, lambda x: sp.exp(x) / x.subs(t, 1)))
    mu_exprs[mu_funcs[n]] = (d_t_mu.replace(sp.exp, lambda x: sp.exp(x) / x.subs(t, 1)))

In [ ]:
K_exprs

In [ ]:
mu = sum(mu_exprs[mu_funcs[i]] for i in range(1, n_max+1))+B
O = formating_rotating(sp.expand(mu * sp.exp(+sp.I * w_d * t)))
mu_exp = grouping_exponential(O)

In [ ]:
decomp={}
freqs_int = [w_p, 3*w_p, 2*w_d-3*w_p,2*w_d-1*w_p,2*w_d+w_p, 3*w_d/2-5*w_p/2, 3*w_d/2-w_p/2, 3*w_d/2+3*w_p/2,3*w_d/2+7*w_p/2,w_d/2-7*w_p/2,w_d/2-3*w_p/2,w_d/2+w_p/2,w_d/2+5*w_p/2,w_d/2+9*w_p/2, w_d-4*w_p,w_d-2*w_p,w_d,w_d+2*w_p,w_d+4*w_p]
freqs_int = [f for f in freqs_int]
terms = mu_exp.args
for term in terms:
    exp_obj = list(term.atoms(sp.exp))[0]
    freq = exp_obj.args[0]/(sp.I*t)
    if freq in freqs_int:
        pre_factor = term / exp_obj
        decomp[freq] = pre_factor

decomp_bis = decomp.copy()
decomp_ter = decomp.copy()

In [ ]:
decomp

In [ ]:
from scipy.special import jv, factorial
#numerical values
w_b_num = 7.05*2*np.pi
w_p_num = 0.95*2*np.pi
da_num = 0
db_num = 0

phi_quantum=1/2


w_d_num = w_b_num #+ 0.05*2*np.pi
phi_a_num = 0.11
phi_b_num = 0.2
E_J_num = 37*2*np.pi

kappa_b_num = 1/10*2*np.pi

Z_a_num = 2*phi_quantum**2*phi_a_num**2
Z_b_num = 2*phi_quantum**2*phi_b_num**2

r_num = np.sqrt(Z_b_num/Z_a_num)


epsilon_p_max_num = 0.6*np.pi
g2_max_num = jv(1,epsilon_p_max_num)*E_J_num*phi_a_num**2*phi_b_num # (Leon's paper)

In [ ]:
g2_max_num /2 /np.pi*1e3

In [ ]:
3/(g2_max_num /2 /np.pi*1e3)

## Pour g2=0.1*g_max

In [ ]:
jv(1,0.1165)/jv(1,epsilon_p_max_num)

In [ ]:
epsilon_p_num = 0.1165

g2_num = jv(1,epsilon_p_num)*E_J_num*phi_a_num**2*phi_b_num # (Leon's paper)
epsilon_d_num = 10*g2_num
xi_num = epsilon_d_num*w_b_num/(1j*kappa_b_num*(2*w_b_num+1j*kappa_b_num))#(w_d_num**2-w_b_num**2)

def g_num(n,k):
    return -2*E_J_num*(-1)**((n+k)/2)/factorial(n)*(Z_a_num/(2*phi_quantum**2))**(n/2)*jv(k,epsilon_p_num)

for (key,value) in decomp.items():
    value = value.subs({w_p: w_p_num, w_d: w_d_num, da: da_num, db: db_num, xi: xi_num, r: r_num})
    # We replace the g coefficients with their numerical values
    for n in range(1, n_max+1, 2):
        for k in range(1, k_max+1, 2):
            value = value.subs(sp.symbols(f"g_{n}_{k}", real=True), g_num(n, k))
    polynom = value.as_poly(A, A_st, B, B_st)
    decomp[key] = polynom.as_dict()

freqs_int_num = {f:f.subs({w_p: w_p_num/(2*np.pi), w_d: w_d_num/(2*np.pi)}) for f in freqs_int}
#sort by frequency value
freqs_int_num = {f: f_num for f, f_num in sorted(freqs_int_num.items(), key=lambda x: x[1])}
decomp_sorted = {f: decomp[f] for f in freqs_int_num}

monomial={A : (1,0,0,0), 
          A_st : (0,1,0,0),
          B : (0,0,1,0),
          B_st : (0,0,0,1),
          A*A_st : (1,1,0,0),
          A*B : (1,0,1,0),
          A*B_st : (1,0,0,1),
          A_st*B : (0,1,1,0),
          A_st*B_st : (0,1,0,1),
          B*B_st : (0,0,1,1),
          A**2 : (2,0,0,0),
          A_st**2 : (0,2,0,0),
          B**2 : (0,0,2,0),
          B_st**2 : (0,0,0,2)}
rates = {}
for key, monomi in monomial.items():
    rates[key]=[]
    for dictio in decomp_sorted.values():
        if monomi in dictio:
            rates[key].append(np.abs(dictio[monomi]))
        else:
            rates[key].append(0)

In [ ]:
decomp_sorted

In [ ]:
fig, ax = plt.subplots(figsize=figsize_car)
for key, rate in rates.items():
    if rate != [0]*len(freqs_int_num):
        ax.plot(freqs_int_num.values(), rate, label=f"{key}")
ax.set_yscale('log')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Rate')
#ax.set_ylim(1e-20, 1e2)
ax.legend(loc='upper right', ncol=3)
plt.show()

## Pour g2=0.01*g2_max

In [ ]:
jv(1,0.01163)/jv(1,epsilon_p_max_num)

In [ ]:
epsilon_p_num = 0.01163

g2_num = jv(1,epsilon_p_num)*E_J_num*phi_a_num**2*phi_b_num # (Leon's paper)
epsilon_d_num = 10*g2_num
xi_num = epsilon_d_num*w_b_num/(1j*kappa_b_num*(2*w_b_num+1j*kappa_b_num))#(w_d_num**2-w_b_num**2)

def g_num(n,k):
    return -2*E_J_num*(-1)**((n+k)/2)/factorial(n)*(Z_a_num/(2*phi_quantum**2))**(n/2)*jv(k,epsilon_p_num)


for (key,value) in decomp_bis.items():
    value = value.subs({w_p: w_p_num, w_d: w_d_num, da: da_num, db: db_num, xi: xi_num, r: r_num})
    # We replace the g coefficients with their numerical values
    for n in range(1, n_max+1, 2):
        for k in range(1, k_max+1, 2):
            value = value.subs(sp.symbols(f"g_{n}_{k}", real=True), g_num(n, k))
    polynom = value.as_poly(A, A_st, B, B_st)
    decomp_bis[key] = polynom.as_dict()

freqs_int_num = {f:f.subs({w_p: w_p_num/(2*np.pi), w_d: w_d_num/(2*np.pi)}) for f in freqs_int}
#sort by frequency value
freqs_int_num = {f: f_num for f, f_num in sorted(freqs_int_num.items(), key=lambda x: x[1])}
decomp_sorted_bis = {f: decomp_bis[f] for f in freqs_int_num}

monomial={A : (1,0,0,0), 
          A_st : (0,1,0,0),
          B : (0,0,1,0),
          B_st : (0,0,0,1),
          A*A_st : (1,1,0,0),
          A*B : (1,0,1,0),
          A*B_st : (1,0,0,1),
          A_st*B : (0,1,1,0),
          A_st*B_st : (0,1,0,1),
          B*B_st : (0,0,1,1),
          A**2 : (2,0,0,0),
          A_st**2 : (0,2,0,0),
          B**2 : (0,0,2,0),
          B_st**2 : (0,0,0,2)}
rates_bis = {}
for key, monomi in monomial.items():
    rates_bis[key]=[]
    for dictio in decomp_sorted_bis.values():
        if monomi in dictio:
            rates_bis[key].append(np.abs(dictio[monomi]))
        else:
            rates_bis[key].append(0)

In [ ]:
fig, ax = plt.subplots(figsize=figsize_car)
for key, rate in rates_bis.items():
    if rate != [0]*len(freqs_int_num):
        ax.plot(freqs_int_num.values(), rate, label=f"{key}")
ax.set_yscale('log')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Rate')
#ax.set_ylim(1e-20, 1e2)
ax.legend(loc='upper right', ncol=3)
plt.show()

## Pour g2=0.001*g2_max

In [ ]:
jv(1,0.001163)/jv(1,epsilon_p_max_num)

In [ ]:
epsilon_p_num = 0.001163

g2_num = jv(1,epsilon_p_num)*E_J_num*phi_a_num**2*phi_b_num # (Leon's paper)
epsilon_d_num = 10*g2_num
xi_num = epsilon_d_num*w_b_num/(1j*kappa_b_num*(2*w_b_num+1j*kappa_b_num))#(w_d_num**2-w_b_num**2)

def g_num(n,k):
    return -2*E_J_num*(-1)**((n+k)/2)/factorial(n)*(Z_a_num/(2*phi_quantum**2))**(n/2)*jv(k,epsilon_p_num)

for (key,value) in decomp_ter.items():
    value = value.subs({w_p: w_p_num, w_d: w_d_num, da: da_num, db: db_num, xi: xi_num, r: r_num})
    # We replace the g coefficients with their numerical values
    for n in range(1, n_max+1, 2):
        for k in range(1, k_max+1, 2):
            value = value.subs(sp.symbols(f"g_{n}_{k}", real=True), g_num(n, k))
    polynom = value.as_poly(A, A_st, B, B_st)
    decomp_ter[key] = polynom.as_dict()

freqs_int_num = {f:f.subs({w_p: w_p_num/(2*np.pi), w_d: w_d_num/(2*np.pi)}) for f in freqs_int}
#sort by frequency value
freqs_int_num = {f: f_num for f, f_num in sorted(freqs_int_num.items(), key=lambda x: x[1])}
decomp_sorted_ter = {f: decomp_ter[f] for f in freqs_int_num}

monomial={A : (1,0,0,0), 
          A_st : (0,1,0,0),
          B : (0,0,1,0),
          B_st : (0,0,0,1),
          A*A_st : (1,1,0,0),
          A*B : (1,0,1,0),
          A*B_st : (1,0,0,1),
          A_st*B : (0,1,1,0),
          A_st*B_st : (0,1,0,1),
          B*B_st : (0,0,1,1),
          A**2 : (2,0,0,0),
          A_st**2 : (0,2,0,0),
          B**2 : (0,0,2,0),
          B_st**2 : (0,0,0,2)}
rates_ter = {}
for key, monomi in monomial.items():
    rates_ter[key]=[]
    for dictio in decomp_sorted_ter.values():
        if monomi in dictio:
            rates_ter[key].append(np.abs(dictio[monomi]))
        else:
            rates_ter[key].append(0)

In [ ]:
fig, ax = plt.subplots(figsize=figsize_car)
for key, rate in rates_ter.items():
    if rate != [0]*len(freqs_int_num):
        ax.plot(freqs_int_num.values(), rate, label=f"{key}")
ax.set_yscale('log')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Rate')
#ax.set_ylim(1e-20, 1e2)
ax.legend(loc='upper right', ncol=3)
plt.show()

In [ ]:
rates_total = {}
for key, monomi in monomial.items():
    rates_total[key] = [[],[],[]]
    for dictio in decomp_sorted.values():
        if monomi in dictio:
            rates_total[key][0].append(np.abs(dictio[monomi]))
        else:
            rates_total[key][0].append(0)
    for dictio in decomp_sorted_bis.values():
        if monomi in dictio:
            rates_total[key][1].append(np.abs(dictio[monomi]))
        else:
            rates_total[key][1].append(0)
    for dictio in decomp_sorted_ter.values():
        if monomi in dictio:
            rates_total[key][2].append(np.abs(dictio[monomi]))
        else:
            rates_total[key][2].append(0)

In [ ]:
rates_total.keys()

In [ ]:
rates_total[A*A_st]

In [ ]:
for key, rates in rates_total.items():
    if rates[0] != [0]*len(freqs_int_num):
        fig, ax = plt.subplots(figsize=figsize_car)
        for i in range(3):
            ax.plot(freqs_int_num.values(), rates[i], label="")
        ax.set_yscale('log')
        ax.set_xlabel('Frequency (Hz)')
        ax.set_ylabel('Rate '+r'$\sqrt{\kappa/\kappa_b}$')
        ax.set_title(f"Rates for {key}")
        ax.set_ylim(1e-20, 1)
        #ax.legend(loc='upper right', ncol=3)
        filename = os.path.expanduser(f"~/Desktop/Figures_dissipation/dissipation_rate_xiao_classical_{key}.png")
        plt.savefig(filename)
        plt.show()

# Test

In [ ]:
C = O +(B + xi) * rot_b
C_grouped = grouping_exponential(C)

In [ ]:
frequencies_exp = list(O.atoms(sp.exp))
for f in frequencies_exp:
    temp = sp.simplify(f.args[0]/sp.I/t)
    #if not temp.has(w_d):
        #if not (temp.has(w_d/2) or temp.has(3*w_d/2) or temp.has(-3*w_d/2) or temp.has(-w_d/2) or temp.has(-2*w_d) or temp.has(2*w_d)):
            #display(Math(sp.latex(temp)))

In [ ]:
len(frequencies_exp)

In [ ]:
freqs_int = [w_p, 3*w_p, 2*w_d-3*w_p,2*w_d-1*w_p,2*w_d+w_p, 3*w_d/2-5*w_p/2, 3*w_d/2-w_p/2, 3*w_d/2+3*w_p/2,3*w_d/2+7*w_p/2,w_d/2-7*w_p/2,w_d/2-3*w_p/2,w_d/2+w_p/2,w_d/2+5*w_p/2,w_d/2+9*w_p/2, w_d-4*w_p,w_d-2*w_p,w_d,w_d+2*w_p,w_d+4*w_p]
freqs_int = [-f for f in freqs_int]
for f in frequencies_exp:
    temp = sp.simplify(f.args[0]/sp.I/t)
    if temp in freqs_int:
        prefactor = sp.simplify(x_indep_terms(sp.expand(O*sp.conjugate(f)), t))
        display(Math(sp.latex(temp)))
        display(Math(sp.latex(prefactor)))
        

In [ ]:
A_t = sp.Function('A', complex=True)(t)
B_t = sp.Function('B', complex=True)(t)
A_st_t = sp.Function('A^*', complex=True)(t)
B_st_t = sp.Function('B^*', complex=True)(t)

C_grouped = C_grouped.replace(A, A_t).replace(B, B_t).replace(A_st, A_st_t).replace(B_st, B_st_t)
#C_grouped

In [ ]:
D = sp.diff(C_grouped,B_st_t)*sp.diff(C_grouped,t)

In [ ]:
decay_dict = {}
for terms in C_grouped.args:
    #display(Math(sp.latex(terms)))
    if terms.has(B_st_t):
        exp_part = list(terms.atoms(sp.exp))[0]
        amp = sp.diff(terms, B_st_t) * sp.diff(sp.conjugate(terms).replace(sp.conjugate(A_t),A_st_t).replace(sp.conjugate(B_t),B_st_t).replace(sp.conjugate(A_st_t),A_t).replace(sp.conjugate(B_st_t),B_t), t)
        amp = formating_rotating(amp)
        decay_dict[exp_part.args[0]/sp.I/t] = amp

In [ ]:

#display(Math(sp.latex(decay_dict)))

In [ ]:
#numerical values
w_d_num = 7.05*2*np.pi
w_p_num = 0.35*2*np.pi
da_num = 0
db_num = 0

phi_quantum=1/2
varphi_a = 0.11
Z_a = 2*phi_quantum**2*varphi_a**2


varphi_b = 0.2
Z_b = 2*phi_quantum**2*varphi_b**2

r_num = np.sqrt(Z_b/Z_a)

for key in decay_dict.keys():
    test = decay_dict[key].subs({w_d: w_d_num, w_p: w_p_num, da: da_num, db: db_num, r: r_num})
    #display(Math(sp.latex(test)))

# We try to go to higher orders with smaller ks

In [ ]:
n_max=5
k_max=1

eta_funcs = [0]
eta_exprs = {0:0}
eta_simplified = {0:0}

mu_funcs = [0]
mu_exprs = {0:0}
mu_simplified = {0:0}

K_funcs = [0]
K_exprs = {0:0}
K_simplified = {0:0}

for i in range(1, n_max+1):
    eta = sp.Function(fr'eta^({i})', complex=True)(A, A_st, B, B_st, t)
    eta_simplified[sp.symbols(fr'eta^({i})', complex=True)] = eta
    eta_funcs.append(eta)

    mu = sp.Function(fr'mu^({i})', complex=True)(A, A_st, B, B_st, t)
    mu_simplified[sp.symbols(fr'mu^({i})', complex=True)] = mu
    mu_funcs.append(mu)

    K = sp.Function(fr'K^({i})', complex=True)(A, A_st, B, B_st)
    K_simplified[sp.symbols(fr'K^({i})', complex=True)] = K
    K_funcs.append(K)

g = sp.Matrix((n_max+1)//2, (k_max+1)//2, lambda i, j: sp.symbols(f"g_{2*i+1}_{2*j+1}", real=True))



for n in range(1, n_max+1):
    eq_a = eq_n(n, 'a')
    eq_b = eq_n(n, 'b')
    
    d_A_st_K = x_indep_terms(sp.expand(eq_a), t)
    d_A_K = sp.conjugate(d_A_st_K).replace(sp.conjugate(A),A_st).replace(sp.conjugate(B),B_st).replace(sp.conjugate(A_st),A).replace(sp.conjugate(B_st),B)

    d_B_st_K = x_indep_terms(sp.expand(eq_b), t)
    d_B_K = sp.conjugate(d_B_st_K).replace(sp.conjugate(A),A_st).replace(sp.conjugate(B),B_st).replace(sp.conjugate(A_st),A).replace(sp.conjugate(B_st),B)

    K_0 = sp.integrate(d_A_st_K, A_st) 
    d_A_K_0 = sp.diff(K_0, A)
    K_0 += sp.integrate(d_A_K - d_A_K_0, A)

    d_B_st_K_0 = sp.diff(K_0, B_st)
    K_0 += sp.integrate(d_B_st_K - d_B_st_K_0, B_st)

    d_B_K_0 = sp.diff(K_0, B)
    K_0 += sp.integrate(d_B_K - d_B_K_0, B)

    K_exprs[K_funcs[n]]=sp.simplify(K_0)

    d_t_eta = formating_rotating(sp.expand(eq_a-d_A_st_K)*(-sp.I))
    d_t_mu = formating_rotating(sp.expand(eq_b-d_B_st_K)*(-sp.I))

    eta_exprs[eta_funcs[n]] = (d_t_eta.replace(sp.exp, lambda x: sp.exp(x) / x.subs(t, 1)))
    mu_exprs[mu_funcs[n]] = (d_t_mu.replace(sp.exp, lambda x: sp.exp(x) / x.subs(t, 1)))

In [ ]:
mu = sum(mu_exprs[mu_funcs[i]] for i in range(1, n_max+1))
O = formating_rotating(sp.expand(mu * sp.exp(-sp.I * w_d * t)))

In [ ]:
frequencies_exp = list(O.atoms(sp.exp))
for f in frequencies_exp:
    temp = sp.simplify(f.args[0]/sp.I/t)
    if temp in freqs_int:
        prefactor = sp.simplify(x_indep_terms(sp.expand(O*sp.conjugate(f)), t))
        display(Math(sp.latex(temp)))
        display(Math(sp.latex(prefactor)))

In [ ]:
2.4*3*2*np.sqrt(1e-9)